In [5]:
import pandas as pd
import joblib

In [6]:
model = joblib.load("../models/weekly_model_v4_promotions.pkl")

In [7]:
df_new = pd.read_csv("../data/processed/weekly_sales_v4_promotions.csv")

df_new["week_start"] = pd.to_datetime(df_new["week_start"])
df_new.head()

,year,week,store_id,product_id,units_sold,avg_price,avg_regular_price,avg_discount_pct,promo_intensity,avg_starting_inventory,...,black_friday_week,month_calendar,season,promo_days_in_week,avg_weekly_discount_pct,has_promo_week,promo_type_Bundle,promo_type_Discount,promo_type_Free Accessory,promo_type_Price Slash
0,2020,53,1,1001,2,445838.0,445838.0,0.0,0.0,9.666667,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
1,2021,1,1,1001,7,445838.0,445838.0,0.0,0.0,6.000000,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
2,2021,2,1,1001,6,445838.0,445838.0,0.0,0.0,7.571429,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
3,2021,3,1,1001,5,445838.0,445838.0,0.0,0.0,5.000000,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
4,2021,4,1,1001,8,445838.0,445838.0,0.0,0.0,5.571429,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0


In [8]:
df_new["store_id"] = df_new["store_id"].astype(str)
df_new["product_id"] = pd.to_numeric(df_new["product_id"], errors="coerce").astype("Int64")

df_new = df_new.dropna(subset=["store_id", "product_id", "units_sold"])
df_new["product_id"] = df_new["product_id"].astype(int)

In [9]:
df_new = df_new.sort_values(["store_id", "product_id", "week_start"]).copy()

group_cols = ["store_id", "product_id"]

df_new["lag_1"] = df_new.groupby(group_cols)["units_sold"].shift(1)
df_new["lag_4"] = df_new.groupby(group_cols)["units_sold"].shift(4)

df_new["rolling_mean_4"] = (
    df_new.groupby(group_cols)["units_sold"]
    .shift(1)
    .rolling(4)
    .mean()
)

df_new = df_new.dropna(subset=["lag_1", "lag_4", "rolling_mean_4"]).copy()

In [10]:
df_new_encoded = pd.get_dummies(
    df_new,
    columns=["season", "store_id", "product_id"],
    drop_first=True
)

In [11]:
model_features = model.feature_names_in_

for col in model_features:
    if col not in df_new_encoded.columns:
        df_new_encoded[col] = 0

df_new_encoded = df_new_encoded[model_features]

In [16]:
X_new = df_new_encoded
#y_actual = df_new["units_sold"]

#y_pred = model.predict(X_new)

In [17]:
X_new

,year,week,avg_price,avg_regular_price,avg_discount_pct,promo_intensity,avg_starting_inventory,month,quarter,week_of_year,...,product_id_1111,product_id_1112,product_id_1113,product_id_1114,product_id_1115,product_id_1116,product_id_1117,product_id_1118,product_id_1119,product_id_1120
4,2021,4,445838.0,445838.0,0.0,0.0,5.571429,1,1,4,...,False,False,False,False,False,False,False,False,False,False
5,2021,5,445838.0,445838.0,0.0,0.0,7.428571,2,1,5,...,False,False,False,False,False,False,False,False,False,False
6,2021,6,445838.0,445838.0,0.0,0.0,6.142857,2,1,6,...,False,False,False,False,False,False,False,False,False,False
7,2021,7,445838.0,445838.0,0.0,0.0,7.285714,2,1,7,...,False,False,False,False,False,False,False,False,False,False
8,2021,8,445838.0,445838.0,0.0,0.0,6.285714,2,1,8,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230155,2026,9,740053.0,740053.0,0.0,0.0,3.000000,2,1,9,...,False,False,False,False,False,False,False,False,False,True
230156,2026,10,740053.0,740053.0,0.0,0.0,3.000000,3,1,10,...,False,False,False,False,False,False,False,False,False,True
230157,2026,11,740053.0,740053.0,0.0,0.0,3.000000,3,1,11,...,False,False,False,False,False,False,False,False,False,True
230158,2026,12,740053.0,740053.0,0.0,0.0,3.714286,3,1,12,...,False,False,False,False,False,False,False,False,False,True


In [18]:
for col in df_new_encoded.columns:
    if df_new_encoded[col].dtype == "bool":
        df_new_encoded[col] = df_new_encoded[col].astype(int)

    elif df_new_encoded[col].dtype == "object":
        df_new_encoded[col] = (
            df_new_encoded[col]
            .replace({"True": 1, "False": 0, True: 1, False: 0})
        )
        df_new_encoded[col] = pd.to_numeric(df_new_encoded[col], errors="coerce").fillna(0)

In [19]:
y_actual = df_new["units_sold"]

y_pred = model.predict(X_new)

In [20]:
from sklearn.metrics import mean_absolute_error
import numpy as np

mae = mean_absolute_error(y_actual, y_pred)
wape = np.sum(np.abs(y_actual - y_pred)) / np.sum(y_actual)

print("MAE:", mae)
print("WAPE:", wape)

MAE: 0.8524169312169312
WAPE: 0.051715846632017906


In [21]:
df_results = df_new.copy()
df_results["predicted_units"] = y_pred

df_results[[
    "week_start",
    "store_id",
    "product_id",
    "units_sold",
    "predicted_units"
]].head(20)

,week_start,store_id,product_id,units_sold,predicted_units
4,2021-01-25,1,1001,8,6.99
5,2021-02-01,1,1001,7,6.93
6,2021-02-08,1,1001,7,6.86
7,2021-02-15,1,1001,8,7.80
8,2021-02-22,1,1001,8,7.91
9,2021-03-01,1,1001,8,7.74
10,2021-03-08,1,1001,10,9.07
11,2021-03-15,1,1001,7,7.69
12,2021-03-22,1,1001,7,7.39
13,2021-03-29,1,1001,7,7.02
